# Smart Budget — SageMaker Endpoint

**Ticket:** DATA-1140 · **Endpoint:** `smart-budget-suggestion-endpoint`

Este notebook hace dos cosas:
1. **Crea el endpoint** (Steps 1–4): empaqueta el pipeline, lo sube a S3 y despliega.
2. **Prueba el endpoint** (Steps 5–6): happy path + validación de las 3 reglas de negocio.

> ⚠️ **Solo boto3** — no requiere el SDK de sagemaker ni SKLearnModel.


In [ ]:
import boto3
import json
import os
import shutil
import tarfile
import time
from pathlib import Path

# ─── Sesión ───────────────────────────────────────────────────────────────────
# Studio: usa las credenciales del entorno de ejecución (rol del dominio)
# Local:  usa perfil blossom-dev
try:
    session = boto3.Session()
    identity = session.client('sts').get_caller_identity()
    print("✅ SageMaker Studio")
except Exception:
    session = boto3.Session(profile_name='blossom-dev')
    identity = session.client('sts').get_caller_identity()
    print("✅ Local (blossom-dev)")

ACCOUNT_ID = identity['Account']
REGION     = session.region_name or 'us-east-1'

# Derivar ARN del rol de ejecución desde la identidad actual
_arn = identity['Arn']
if ':assumed-role/' in _arn:
    _role_name = _arn.split(':assumed-role/')[1].split('/')[0]
    ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{_role_name}"
else:
    ROLE_ARN = _arn

# Clientes boto3 — los únicos que necesitamos
s3      = session.client('s3')
sm      = session.client('sagemaker')
runtime = session.client('sagemaker-runtime')

# Constantes del endpoint
ENDPOINT_NAME = 'smart-budget-suggestion-endpoint'
S3_BUCKET     = 'blossom-analytics-datalake-dev'
S3_KEY        = 'smart_budget/endpoint/v1/model.tar.gz'
S3_URI        = f's3://{S3_BUCKET}/{S3_KEY}'

# Contenedor SKLearn 1.2-1 por región (sin sagemaker SDK)
_SKLEARN_IMAGES = {
    'us-east-1': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3',
    'us-east-2': '257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3',
    'us-west-2': '246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3',
    'eu-west-1': '141502667606.dkr.ecr.eu-west-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3',
}
CONTAINER_IMAGE = _SKLEARN_IMAGES.get(REGION, _SKLEARN_IMAGES['us-east-1'])

print(f"Account : {ACCOUNT_ID}")
print(f"Region  : {REGION}")
print(f"Role    : {ROLE_ARN}")
print(f"Image   : {CONTAINER_IMAGE}")


---
## Step 1 — Empaquetar model.tar.gz

Estructura del tarball:
```
model.tar.gz
├── inference.py        ← entry_point
├── smart_budget/       ← paquete src/smart_budget/
└── data/
    ├── smart_budget_synthetic.csv
    ├── test_internal.csv
    └── test_external.csv
```


In [ ]:
REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())

SRC_INFERENCE    = REPO_ROOT / 'src' / 'api' / 'inference.py'
SRC_SMART_BUDGET = REPO_ROOT / 'src' / 'smart_budget'
DATA_DIR         = REPO_ROOT / 'data' / 'dough'
ARTIFACTS_DIR    = REPO_ROOT / 'notebooks' / 'model_artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Staging limpio
staging = ARTIFACTS_DIR / 'staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

# Copiar inference.py y paquete smart_budget
shutil.copy(SRC_INFERENCE, staging / 'inference.py')
shutil.copytree(SRC_SMART_BUDGET, staging / 'smart_budget')

# Copiar CSVs de datos
data_staging = staging / 'data'
data_staging.mkdir()
for name in ['smart_budget_synthetic.csv']:
    src = DATA_DIR / name
    if src.exists(): shutil.copy(src, data_staging / name)
    else: print(f"⚠️  No encontrado: {src}")
for name in ['test_internal.csv', 'test_external.csv']:
    src = DATA_DIR / 'test' / name
    if src.exists(): shutil.copy(src, data_staging / name)
    else: print(f"⚠️  No encontrado: {src}")

# Crear tarball
tarball_path = ARTIFACTS_DIR / 'model.tar.gz'
with tarfile.open(tarball_path, 'w:gz') as tar:
    for item in staging.rglob('*'):
        if item.is_file():
            tar.add(item, arcname=item.relative_to(staging))

print(f"✅ model.tar.gz: {tarball_path}  ({tarball_path.stat().st_size / 1024:.1f} KB)")


---
## Step 2 — Subir model.tar.gz a S3

In [ ]:
s3.upload_file(str(tarball_path), S3_BUCKET, S3_KEY)
print(f"✅ Subido a: {S3_URI}")


---
## Step 3 — Crear modelo en SageMaker

In [ ]:
MODEL_NAME = f'smart-budget-model-{int(time.time())}'

sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={
        'Image': CONTAINER_IMAGE,
        'ModelDataUrl': S3_URI,
        'Environment': {
            'SAGEMAKER_PROGRAM': 'inference.py',
            'SAGEMAKER_SUBMIT_DIRECTORY': S3_URI,
        },
    },
)
print(f"✅ Modelo creado: {MODEL_NAME}")


---
## Step 4 — Crear endpoint config y desplegar

In [ ]:
CONFIG_NAME = f'smart-budget-config-{int(time.time())}'

sm.create_endpoint_config(
    EndpointConfigName=CONFIG_NAME,
    ProductionVariants=[{
        'VariantName': 'AllTraffic',
        'ModelName': MODEL_NAME,
        'InitialInstanceCount': 1,
        'InstanceType': 'ml.m5.large',
        'InitialVariantWeight': 1,
    }],
)

# Crear o actualizar endpoint
existing = [e['EndpointName'] for e in sm.list_endpoints()['Endpoints']]
if ENDPOINT_NAME in existing:
    sm.update_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)
    print(f"🔄 Actualizando endpoint: {ENDPOINT_NAME}")
else:
    sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)
    print(f"🚀 Creando endpoint: {ENDPOINT_NAME}")

# Esperar hasta que esté InService
print("Esperando... (puede tardar 3-5 minutos)")
waiter = sm.get_waiter('endpoint_in_service')
waiter.wait(EndpointName=ENDPOINT_NAME, WaiterConfig={'Delay': 15, 'MaxAttempts': 40})
print(f"✅ Endpoint listo: {ENDPOINT_NAME}")


---
## Step 5 — Probar el endpoint (happy path)

Invoca el endpoint con una cuenta y categoría que tienen datos.


In [ ]:
def invoke(payload: dict) -> dict:
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Body=json.dumps(payload),
    )
    return json.loads(response['Body'].read().decode('utf-8'))

# TC-1: Happy path — EXT2 + Food & Dining tiene datos
result = invoke({'idaccount': 'EXT2', 'defaultcategory': 'Food & Dining', 'period_id': '2026-05'})
print(json.dumps(result, indent=2))


---
## Step 6 — Validar las 3 reglas de negocio

| # | Condición | Respuesta esperada |
|---|---|---|
| Regla 1 | Cuenta no existe | `ModelError` (error 400) |
| Regla 2 | Categoría inválida | `ModelError` (error 400) |
| Regla 3 | Sin datos para ese período | `suggested_amount: null` (HTTP 200) |


In [ ]:
import botocore

# Regla 1 — Cuenta no existe → ModelError
try:
    invoke({'idaccount': 'CUENTA_INEXISTENTE', 'defaultcategory': 'Groceries', 'period_id': '2026-05'})
    print("❌ Regla 1 FALLÓ: debería haber dado error")
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print("✅ Regla 1 OK — cuenta inexistente → error")

# Regla 2 — Categoría inválida → ModelError
try:
    invoke({'idaccount': 'EXT2', 'defaultcategory': 'CategoriaFalsa', 'period_id': '2026-05'})
    print("❌ Regla 2 FALLÓ: debería haber dado error")
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print("✅ Regla 2 OK — categoría inválida → error")

# Regla 3 — SYN001 + Groceries: cuenta existe pero sin datos → null
r = invoke({'idaccount': 'SYN001', 'defaultcategory': 'Groceries', 'period_id': '2026-05'})
assert r['suggested_amount'] is None, f"Esperaba null, recibí: {r['suggested_amount']}"
print(f"✅ Regla 3 OK — sin datos → null  ({r.get('display_label')})")


---
## ⚠️ Borrar endpoint cuando termines

Los endpoints generan costo por hora mientras están activos.


In [ ]:
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
print(f"✅ Endpoint eliminado: {ENDPOINT_NAME}")
